# Backtest de estrategias intradia - oro y BrentDiversifica | Magdalena MontoyaPrueba varias estrategias sobre **tus** ticks de Pepperstone, con **tu** spread,en un solo motor para que compitan en igualdad.**El orden importa y no es decorativo:** primero corren los controles del motor.Si un control falla, los resultados de abajo no significan nada y hay que parar.Un backtest sin controles es una opinion con graficos.### Antes de empezar1. En el VPS: `python exportar_ticks.py --dias 60`2. Sube los `.parquet` a tu Google Drive, a una carpeta `diversifica`3. Ejecuta las celdas de arriba a abajo

## 1. Montar Drive y traer el codigo

In [ ]:
from google.colab import drivedrive.mount('/content/drive')

In [ ]:
# El codigo viene del repo publico, no pegado en el notebook: asi el motor# que corre aqui es exactamente el mismo que esta versionado, y si lo# cambiamos no quedan dos copias distintas dando resultados distintos.import osif os.path.exists('execution-analysis'):    !cd execution-analysis && git pull -qelse:    !git clone -q https://github.com/magdamontoya/execution-analysis.git%cd /content/execution-analysis!ls *.py

## 2. Los controles del motor. Si esto falla, paramos

In [ ]:
!python test_bt.py

## 3. Encontrar los archivos de ticksBusca sola los `.parquet` en tu Drive, asi no hay que acertar la ruta a mano.

In [ ]:
import globcandidatos = sorted(glob.glob('/content/drive/MyDrive/**/ticks_*.parquet', recursive=True))candidatos += sorted(glob.glob('/content/drive/MyDrive/**/ticks_*.csv.gz', recursive=True))for c in candidatos:    print('%8.1f MB  %s' % (os.path.getsize(c)/1e6, c))if not candidatos:    print('No encuentro ningun ticks_*.parquet en tu Drive.')    print('Subelos a una carpeta de MyDrive y vuelve a correr esta celda.')

In [ ]:
# Empareja cada instrumento con su archivo por el nombredef buscar(clave):    for c in candidatos:        if clave.lower() in os.path.basename(c).lower():            return c    return NoneARCHIVOS = {    'ORO':   buscar('XAUUSD') or buscar('GOLD'),    'BRENT': buscar('Brent')  or buscar('UKOIL'),}ARCHIVOS = {k: v for k, v in ARCHIVOS.items() if v}ARCHIVOS

## 4. Ticks a barrasEl punto medio `(bid+ask)/2` da el precio y el spread se guarda aparte, por barra.Usar solo el bid meteria el spread dentro de la serie y luego se cobraria otravez al ejecutar: se pagaria dos veces y todo pareceria peor de lo que es.

In [ ]:
import numpy as npimport pandas as pdfrom bt_core import cargar_ticks, hacer_barras, correr, estadisticas, tablafrom estrategias import CATALOGOMINUTOS = 5BARRAS = {}for nombre, ruta in ARCHIVOS.items():    t = cargar_ticks(ruta)    b = hacer_barras(t, minutos=MINUTOS)    BARRAS[nombre] = b    print('%-7s %s ticks -> %s barras de %d min' % (        nombre, format(len(t), ','), format(len(b), ','), MINUTOS))    print('        %s a %s' % (b['time'].iloc[0], b['time'].iloc[-1]))    print('        spread mediano %.5f = %.3f%% del rango mediano de barra' % (        b['spread'].median(),        100 * b['spread'].median() / (b['high'] - b['low']).median()))

## 5. Correr las seis estrategias`ALEATORIA` no es un relleno: es el **suelo**. Entra al azar con la mismagestion de stop y objetivo que las demas. Como el motor supone que el stopsalta primero cuando stop y objetivo caben en la misma barra, el suelo no escero sino un numero negativo. Una estrategia solo existe si le gana a esa fila.

In [ ]:
RESULTADOS = {}OPERACIONES = {}for inst, b in BARRAS.items():    filas = []    # primero las ideas    for nombre, fn in CATALOGO.items():        if nombre == 'ALEATORIA':            continue        sen, st, ob = fn(b)        tope = 10 ** 6 if nombre == 'comprar_mantener' else 48        ops = correr(b, sen, st, ob, max_barras=tope, etiqueta=nombre)        OPERACIONES[(inst, nombre)] = ops        filas.append((nombre, estadisticas(ops)))    # y ahora el azar, con la frecuencia de la que MAS opera: comparar una    # estrategia de 60 operaciones con un azar de 600 no compara nada    n_obj = max([len(o) for (i2, k), o in OPERACIONES.items()                 if i2 == inst and k != 'comprar_mantener'] + [200])    sen, st, ob = CATALOGO['ALEATORIA'](b, n_senales=n_obj)    ops = correr(b, sen, st, ob, max_barras=48, etiqueta='ALEATORIA')    OPERACIONES[(inst, 'ALEATORIA')] = ops    filas.append(('ALEATORIA', estadisticas(ops)))    RESULTADOS[inst] = dict(filas)    tabla(filas, titulo=inst)

## 6. La comparacion que de verdad decideCada estrategia contra el azar **con la misma gestion**. La columna que importaes la ultima: cuanto le saca al azar por operacion.

In [ ]:
for inst, res in RESULTADOS.items():    base = res.get('ALEATORIA', {})    if not base.get('n'):        continue    print('\n%s   (el azar saca %+.5f por operacion)' % (inst, base['expectativa']))    print('  %-20s %8s %11s %11s' % ('estrategia', 'n', 'expect.', 'vs azar'))    print('  ' + '-' * 54)    for nombre, e in sorted(res.items(),                            key=lambda kv: -(kv[1].get('expectativa') or -9e9)):        if not e.get('n'):            continue        print('  %-20s %8d %+11.5f %+11.5f' % (            nombre, e['n'], e['expectativa'], e['expectativa'] - base['expectativa']))

## 7. Prueba de robustez: la misma idea en otros marcos temporalesUna ventaja real no desaparece al cambiar de 5 a 15 minutos. Si una estrategiasolo funciona en un marco y en uno solo, lo mas probable es que ese numero seael resultado de haber probado muchas combinaciones hasta que una salio bien.Esto tarda unos minutos.

In [ ]:
filas = []for inst, ruta in ARCHIVOS.items():    t = cargar_ticks(ruta)    for mins in [5, 15, 30]:        b = hacer_barras(t, minutos=mins)        for nombre, fn in CATALOGO.items():            if nombre == 'comprar_mantener':                continue            sen, st, ob = fn(b)            ops = correr(b, sen, st, ob, max_barras=48, etiqueta=nombre)            e = estadisticas(ops)            if e.get('n', 0) < 20:                continue            filas.append({'instrumento': inst, 'minutos': mins,                          'estrategia': nombre, 'n': e['n'],                          'expectativa': e['expectativa'], 't': e['t'],                          'spreads': e['en_spreads']})rob = pd.DataFrame(filas)piv = rob.pivot_table(index=['instrumento', 'estrategia'],                      columns='minutos', values='t')print('t-stat por marco temporal (necesitas > 2 en VARIOS, no en uno):')piv.round(2)

## 8. Guardar en DriveLos resultados se quedan en tu Drive, no en el notebook: Colab borra la maquinay con ella todo lo que no se haya guardado.

In [ ]:
SALIDA = '/content/drive/MyDrive/diversifica/resultados'os.makedirs(SALIDA, exist_ok=True)sello = pd.Timestamp.now().strftime('%Y%m%d_%H%M')rob.to_csv('%s/robustez_%s.csv' % (SALIDA, sello), index=False)todas = pd.concat([o.assign(instrumento=k[0]) for k, o in OPERACIONES.items()                   if len(o)], ignore_index=True)todas.to_csv('%s/operaciones_%s.csv' % (SALIDA, sello), index=False)print('guardado en %s' % SALIDA)print('%d operaciones en total' % len(todas))

## Como leer esto**Lo que normalmente va a pasar:** ninguna estrategia pasa el filtro. Eso no esun fracaso del notebook, es el resultado mas frecuente y es informacion que valedinero — concretamente, el dinero que no pierdes operandola en vivo.**Si alguna pasa**, antes de ponerle un euro:1. ¿Le gana al azar con la misma gestion, no solo a cero?2. ¿Aguanta en 5, 15 y 30 minutos, o solo en uno?3. ¿Aguanta en los dos instrumentos?4. ¿Cuantos spreads gana por operacion? Por debajo de 1, el negocio es del broker.5. ¿Sigue ahi si partes los 60 dias en dos mitades y mides cada una?El punto 5 todavia no esta en el notebook. Es el siguiente que hay que anadir, yes el que mas estrategias mata.